In [2]:
import pandas as pd
from pathlib import Path

In [5]:
interim_df = pd.read_parquet("../data/interim/lastfm/lastfm_scrobbles_merged.parquet")

In [6]:
interim_df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
559444,Perfume Genius,No Front Teeth,NaN,2026
559445,Perfume Genius,No Front Teeth,NaN,2026
559446,Perfume Genius,No Front Teeth,NaN,2026
559447,Perfume Genius,No Front Teeth,NaN,2026


In [10]:
# 1. Drop rows with missing timestamps
df = interim_df.dropna(subset=["timestamp"])
df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026
558999,Stellardrone,Tranquility,1767472374,2026


In [9]:
# 2. Convert timestamps to datetime and extract year and month
df["timestamp"] = pd.to_numeric(df["timestamp"], errors="coerce")
df["timestamp"] = df["timestamp"].astype(int)
df["played_at"] = pd.to_datetime(df["timestamp"], unit="s")
df["year"] = df["played_at"].dt.year
df["month"] = df["played_at"].dt.to_period("M")
df

,artist,track,timestamp,year_file,played_at,year,month
0,Oasis,Don't Look Back in Anger,1199052426,2007,2007-12-30 22:07:06,2007,2007-12
1,Oasis,Wonderwall,1199052167,2007,2007-12-30 22:02:47,2007,2007-12
2,Oasis,Don't Look Back in Anger,1199051867,2007,2007-12-30 21:57:47,2007,2007-12
3,Oasis,Wonderwall,1199051609,2007,2007-12-30 21:53:29,2007,2007-12
4,Incubus,Aqueous Transmission,1199051161,2007,2007-12-30 21:46:01,2007,2007-12
...,...,...,...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026,2026-01-03 20:45:19,2026,2026-01
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026,2026-01-03 20:41:22,2026,2026-01
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026,2026-01-03 20:38:18,2026,2026-01
558999,Stellardrone,Tranquility,1767472374,2026,2026-01-03 20:32:54,2026,2026-01


In [15]:
# 4. Sanity check
print(f"{df.shape} records after processing.")
print(f"Number of empty rows {df.isna().sum()}")
print(f"Number of unique artists in the df {df.artist.nunique()}")
print(f"Number of unique tracks in the df {df.track.nunique()}")
print(f"Top rows {df.head()}")

(550151, 4) records after processing.
Number of empty rows artist       0
track        0
timestamp    0
year_file    0
dtype: int64
Number of unique artists in the df 13837
Number of unique tracks in the df 69106
Top rows     artist                     track   timestamp  year_file
0    Oasis  Don't Look Back in Anger  1199052426       2007
1    Oasis                Wonderwall  1199052167       2007
2    Oasis  Don't Look Back in Anger  1199051867       2007
3    Oasis                Wonderwall  1199051609       2007
4  Incubus      Aqueous Transmission  1199051161       2007


In [18]:
# 5. Deduplication
df.duplicated(subset=["artist", "track", "timestamp"]).sum()
df = df.drop_duplicates(subset=["artist", "track", "timestamp"])
df

,artist,track,timestamp,year_file
0,Oasis,Don't Look Back in Anger,1199052426,2007
1,Oasis,Wonderwall,1199052167,2007
2,Oasis,Don't Look Back in Anger,1199051867,2007
3,Oasis,Wonderwall,1199051609,2007
4,Incubus,Aqueous Transmission,1199051161,2007
...,...,...,...,...
558996,Alaskan Tapes,Wait,1767473119,2026
558997,Brian Eno,An Ending (Ascent) - Remastered 2005,1767472882,2026
558998,Michael Andrews,Boy Moves The Sun,1767472698,2026
558999,Stellardrone,Tranquility,1767472374,2026


In [32]:
# 6. Cleaning track and artist names
import re

def clean_text_artist(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # Remove content in parentheses
    text = re.sub(r"\(.*?\)", "", text)
    
    # Remove weird suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # Remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # Remove special signs and punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # Remove and trim extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [33]:
def clean_text_track(text):
    if text is None:
        return None
    
    text = text.lower()
    
    # Remove content in parentheses
    text = re.sub(r"\(.*?\)", "", text)
    
    # Remove "feat / ft." as separate words"
    text = re.sub(r"\b(feat|ft)\.?\b.*", "", text)
    
    # Remove weird suffixes
    text = re.sub(r"\d{2,}[a-z]{3,}\d*$", "", text)
    
    # Remove " - something"
    text = re.sub(r"\s-\s.*", "", text)

    # Remove special signs and punctuation
    text = re.sub(r"[^a-z0-9\s]", "", text)

    # Remove and trim extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [34]:
df["artist_clean"] = df["artist"].apply(clean_text_artist)
df["track_clean"] = df["track"].apply(clean_text_track)

In [35]:
df[["artist", "artist_clean", "track", "track_clean"]].head(50)

,artist,artist_clean,track,track_clean
0,Oasis,oasis,Don't Look Back in Anger,dont look back in anger
1,Oasis,oasis,Wonderwall,wonderwall
2,Oasis,oasis,Don't Look Back in Anger,dont look back in anger
3,Oasis,oasis,Wonderwall,wonderwall
4,Incubus,incubus,Aqueous Transmission,aqueous transmission
5,Incubus,incubus,Under My Umbrella12DSGMS192,under my umbrella
6,Incubus,incubus,Are You In11DSGMS192,are you in
7,Incubus,incubus,Have You Ever ?,have you ever
8,Incubus,incubus,Echo,echo
9,Incubus,incubus,Warning,warning


In [37]:
print("Before:", df["track"].nunique())
print("After:", df["track_clean"].nunique())

Before: 69106
After: 57030


In [ ]:
#TODO: fix cleaning of tracks and artists so that parts of titles are not deleted like deftones - change (in the house of flies)